# Retinal Baseline Training

## Purpose
This notebook trains a baseline retinal disease classifier in Google Colab using the repo-managed training pipeline. It is intended for the 5-class baseline only, not the full 9-class problem, and it supports provisional training on unreviewed manifests when explicitly enabled in the training command.

## Requirements
- Google Colab runtime with GPU enabled
- Google Drive mounted in the notebook
- Repo available at `MyDrive/retina_project/repo`
- Dataset available at `MyDrive/retina_project/data`
- Split manifests available under `data/splits/`
- Raw retinal images available under `data/raw/original/`

## Expected Drive Layout
```text
MyDrive/
  retina_project/
    repo/
      training/
      colab/
      ...
    data/
      splits/
        train.csv
        val.csv
        test.csv
      raw/
        original/
          ...
    runs/
```

## What This Notebook Does
- Mounts Google Drive
- Installs the Python dependencies needed for training
- Verifies that the repo, manifests, and raw image paths match the expected layout
- Runs the baseline training job in Colab
- Writes checkpoints and evaluation artifacts into `runs/`

## Outputs
After a successful run, the output directory will contain:
- `best_model.pt`
- `last_model.pt`
- `metrics.json`
- `history.csv`
- `label_map.json`
- `run_config.json`
- `val_confusion_matrix.png`
- `test_confusion_matrix.png`

## Current Defaults
- Model: `efficientnet_b0`
- Image size: `224`
- Target classes: the 5-class baseline
- Provisional mode is only active when explicitly requested in the training command

## Important Caveats
- Provisional mode uses unreviewed manifests and should be treated as a pipeline baseline, not a final dataset run
- This notebook supports a research workflow, not a clinical or production diagnostic system
- Test metrics are only meaningful if the Drive paths, manifests, and raw image files are correct before training starts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch torchvision timm scikit-learn pandas pillow matplotlib

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU not detected. In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, reconnect, and rerun the notebook.'
    )

print('GPU is available.')
print('Device:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/retina_project')
REPO_ROOT = PROJECT_ROOT / 'repo'
DATA_ROOT = PROJECT_ROOT
TRAIN_CSV = DATA_ROOT / 'data' / 'splits' / 'train.csv'
VAL_CSV = DATA_ROOT / 'data' / 'splits' / 'val.csv'
TEST_CSV = DATA_ROOT / 'data' / 'splits' / 'test.csv'
RAW_ROOT = DATA_ROOT / 'data' / 'raw' / 'original'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'baseline_efficientnet_b0'

print(PROJECT_ROOT)
print(REPO_ROOT)
print(DATA_ROOT)
print(TRAIN_CSV)
print(VAL_CSV)
print(TEST_CSV)
print(RAW_ROOT)
print(RUN_ROOT)

In [ ]:
import pandas as pd

required_paths = {
    'repo_root': REPO_ROOT,
    'training_script': REPO_ROOT / 'training' / 'train_colab.py',
    'train_csv': TRAIN_CSV,
    'val_csv': VAL_CSV,
    'test_csv': TEST_CSV,
    'raw_root': RAW_ROOT,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
for name, path in required_paths.items():
    print(f'{name}: {path} -> {path.exists()}')

if missing:
    raise FileNotFoundError('Missing required paths: ' + ', '.join(missing))

sample = pd.read_csv(TRAIN_CSV, nrows=3)
print('\nTrain CSV columns:', list(sample.columns))
print(sample[['image_id', 'file_path', 'class_name']])

sample_paths = [DATA_ROOT / path for path in sample['file_path'].tolist()]
for path in sample_paths:
    print(f'sample_image: {path} -> {path.exists()}')

if not all(path.exists() for path in sample_paths):
    raise FileNotFoundError('At least one sample image path from the manifest does not resolve under DATA_ROOT.')

print('\nPath verification passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m training.train_colab \
  --train-csv {TRAIN_CSV} \
  --val-csv {VAL_CSV} \
  --test-csv {TEST_CSV} \
  --data-root {DATA_ROOT} \
  --output-dir {RUN_ROOT} \
  --model efficientnet_b0 \
  --img-size 224 \
  --batch-size 32 \
  --epochs 15 \
  --lr 3e-4 \
  --weight-decay 1e-4 \
  --num-workers 2 \
  --allow-provisional-data \
  --run-test-after-training

In [ ]:
import json
from IPython.display import Image, display

OUTPUT_DIR = RUN_ROOT.with_name(RUN_ROOT.name + '_provisional') if (RUN_ROOT.with_name(RUN_ROOT.name + '_provisional')).exists() else RUN_ROOT

with open(OUTPUT_DIR / 'metrics.json', 'r', encoding='utf-8') as handle:
    metrics = json.load(handle)

metrics

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'val_confusion_matrix.png')))
display(Image(filename=str(OUTPUT_DIR / 'test_confusion_matrix.png')))